# Mollier h,x-Diagram

This example demonstrates how to use `plot_mollier_hx` to create
psychrometric charts (Mollier h,x-diagrams) for visualising the state
of moist air.

All plots are **interactive** — hover over data points to see details.

## Empty diagram (coordinate grid only)

Pass `comfort_zone=False` to show only the iso-lines without a comfort zone.

In [ ]:
from pyedautils.plots import plot_mollier_hx
from IPython.display import HTML

html = plot_mollier_hx(comfort_zone=False)
HTML(html)

## With default comfort zone

Calling `plot_mollier_hx()` without arguments adds the default comfort zone
(T: 20–26 °C, φ: 30–65 %, x: 0–11.5 g/kg).

In [ ]:
html = plot_mollier_hx()
HTML(html)

## Diagram with measured data

Pass a DataFrame with columns `[timestamp, humidity, temperature]`
(humidity in %, temperature in °C) to overlay data points.
Points are automatically colour-coded by season.

In [ ]:
import pandas as pd
from importlib import resources

# Anonymised hourly room climate (temperature + humidity) from a real sensor.
data_path = resources.files("pyedautils") / "data" / "room_climate_sample.csv"
df = pd.read_csv(data_path)
df.head()

In [ ]:
html = plot_mollier_hx(data=df)
HTML(html)

## Frequency contour lines ("risk hours")

Set `show_frequency=True` to overlay cumulative-frequency contour lines on the
scatter. Each line labelled *N* encloses all but *N* units of the year, so the
outer lines mark the rare extremes and the inner lines the typical core.

- `frequency_unit` — `"hours"` (default; line *N* = *N* hours/year outside) or
  `"days"` (line *N* = *N* distinct calendar days/year with at least one hour
  outside).
- `frequency_smoothing` — line rounding (~0.6 angular … ~9 very round, default
  `4.0`). Purely geometric: it rounds the curves but never enlarges the
  enclosed region.

Hover a line to read its value.

In [ ]:
# Hours mode (default): each line N encloses all but N hours/year
html = plot_mollier_hx(data=df, show_frequency=True)
HTML(html)

In [ ]:
# Days mode: each line N = N distinct calendar days/year with >=1 hour outside
html = plot_mollier_hx(data=df, show_frequency=True, frequency_unit="days")
HTML(html)

## Customisation

You can adjust the pressure (e.g. for higher altitude) and change the
comfort zone.

In [ ]:
html = plot_mollier_hx(
    data=df,
    pressure=95000.0,  # ~500 m altitude
    comfort_zone={
        "temperature": (18, 24),
        "rel_humidity": (0.20, 0.70),
        "abs_humidity": (0, 0.012),
    },
)
HTML(html)

## Custom axis ranges

Use `domain_x` and `domain_y` to zoom into a specific region of the diagram.
`domain_x` controls the absolute humidity range (kg/kg) and `domain_y`
controls the y-coordinate range (≈ temperature at x=0).

In [ ]:
html = plot_mollier_hx(
    data=df,
    domain_x=(0.002, 0.014),  # absolute humidity 2–14 g/kg
    domain_y=(10.0, 35.0),    # y-range ≈ 10–35 °C
)
HTML(html)

## Toggling individual curve families

Each iso-line family can be switched off individually (all are on by
default). Hiding a family removes **everything** belonging to it — the
lines, their value labels and the axis-edge caption:

- `show_temperature` — blue iso-temperature lines (the temperature y-axis
  itself always stays, it is the chart's coordinate base).
- `show_density` — grey iso-density curves + `"Density ρ [kg/m³]"`.
- `show_rel_humidity` — the φ curves and their labels (the 100 %
  saturation curve / chart boundary is always drawn).
- `show_enthalpy` — iso-enthalpy lines + `"Enthalpy h [kJ/kg]"`.
- `show_abs_humidity` — vertical iso-lines of constant absolute humidity (at the 1 g/kg major-tick positions).

The caption centred below the bottom axis is controlled by
`x_axis_title` (default `"absolute water content x [g/kg]"`; pass `""` to omit).

Seasonal scatter labels default to English. Pass a mapping via `season_labels` (e.g. `pyedautils.plots._constants._SEASON_LABELS_DE`) for German.

In [ ]:
# Hide density and enthalpy; keep temperature and humidity
html = plot_mollier_hx(
    data=df,
    show_density=False,
    show_enthalpy=False,
)
HTML(html)

## Warning bands, altitude & pressure

Add up to two extra zones drawn as **dashed outlines** (no fill) on top of the
green comfort zone via `comfort_zone_orange` and `comfort_zone_red`. Each takes
the same absolute-bounds dict as `comfort_zone` — `temperature` (°C),
`rel_humidity` (0–1) and optional `abs_humidity` (kg/kg, e.g. `(0, 0.012)` for a
12 g/kg ceiling) — plus an optional `label` (legend caption) and `color`.

A temperature *difference* is expressed in kelvin, so label the bands in `K`.

The altitude (m a.s.l.) and pressure (hPa) are written in the top-right corner.
Pass `altitude` explicitly, or leave it out to derive it from `pressure` via the
inverse ISA barometric formula.

In [ ]:
from pyedautils._mollier import pressure_from_altitude

p = pressure_from_altitude(450)
html = plot_mollier_hx(
    data=df,
    pressure=p,
    altitude=450,
    comfort_zone={"temperature": (20, 26), "rel_humidity": (0.30, 0.65)},
    comfort_zone_orange={"temperature": (19, 27), "rel_humidity": (0.25, 0.70),
                         "abs_humidity": (0, 0.012), "label": "± 1 K / 5 %"},
    comfort_zone_red={"temperature": (17.5, 28.5), "rel_humidity": (0.20, 0.75),
                      "abs_humidity": (0, 0.013), "label": "± 2.5 K / 10 %"},
)
HTML(html)

## Convention: classical vs. Glück

The y-axis transformation has two valid normalisations. `'classical'`
(default) follows Mollier 1923 / Recknagel — enthalpy per kg of dry
air, isotherms tilt slightly **up** with x. `'glueck'` follows the
Glück reference — enthalpy per kg of moist air, isotherms tilt slightly
**down**. Physical quantities (T, φ, ρ, h) are identical under either
convention; only the (x, y) parametrisation differs.

In [ ]:
html = plot_mollier_hx(data=df, convention='glueck')
HTML(html)

## Latest-record overlay

When `data` is supplied, the row with the newest timestamp is overlaid as a
black circle on top of the seasonal scatter — you've already seen it in the
diagrams above. The overlay is controlled by two parameters:

- `highlight_latest` (bool, default `True`) — show or hide the overlay.
- `highlight_color` (str | None, default `'black'`) — any CSS colour, or
  `None` to fall back to the row's season colour.

In [ ]:
# Disable the latest-row overlay
html = plot_mollier_hx(data=df, highlight_latest=False)
HTML(html)

In [ ]:
# Custom highlight colour
html = plot_mollier_hx(data=df, highlight_color='#e53935')
HTML(html)

## Process chain (psychrosim-style)

Use the `states=` argument to overlay a sequence of psychrometric state
points joined by arrows — a multi-process simulation in the spirit of
[psychrosim.com](https://www.psychrosim.com/). Build each state with the
`state(...)` factory in `pyedautils._mollier` and chain them through the
process functions (`heat`, `cool`, `humidify_adiabatic`, …).

Below we also show how to derive the ambient pressure from a given
altitude with `pressure_from_altitude` (ISA barometric formula, valid up
to ~5000 m). The example below assumes a building at 450 m a.s.l.

In [ ]:
from pyedautils._mollier import (
    state, heat, humidify_adiabatic, heat_recovery,
    pressure_from_altitude,
)

# 450 m a.s.l. → atmospheric pressure via ISA
p = pressure_from_altitude(450)
print(f'p at 450 m a.s.l.: {p:.0f} Pa')

# Winter: outdoor -5 °C / 80 % RH, extract 22 °C / 40 % RH, 1500 m³/h
s1 = state(t=-5, phi=0.80, p=p, volume_flow=1500)
extract = state(t=22, phi=0.40, p=p, volume_flow=1500)

s2, _ = heat_recovery(s1, extract, eps_sensible=0.75)
s3, _ = heat(s2, t_out=21)
s4, _ = humidify_adiabatic(s3, phi_out=0.45)

html = plot_mollier_hx(
    states=[s1, s2, s3, s4],
    labels=['Outdoor', 'After HR', 'After heater', 'After humidifier'],
    pressure=p,
    domain_x=(0.0, 0.012),
    domain_y=(-10, 30),
    comfort_zone=False,
)
HTML(html)